# Philippine Law Assistant — Google Colab (GPU)

Run the **same RAG + law tutor prompt** as the project backend, but with **GPU inference** (usually much faster than on-device CPU).

## Before you run
1. **Runtime → Change runtime type → Hardware accelerator → GPU** (T4 or better).
2. **Backend code** — the next cell clones **`daveyous07/phillipinelawai`** by default (includes `backend/`). Override with env `PHLAW_REPO_URL` or upload a zip and set `BACKEND_ROOT` in that cell.

## Model
Default: **`Qwen/Qwen2.5-1.5B-Instruct`** (no Llama license gate). Change `MODEL_ID` for another small instruct model.

## 1) Install dependencies

In [ ]:
%pip install -q torch transformers accelerate sentencepiece protobuf safetensors

In [ ]:
import torch

assert torch.cuda.is_available(), "Enable GPU: Runtime → Change runtime type → GPU"
print("GPU:", torch.cuda.get_device_name(0))

## 2) Point Python at `backend/` (RAG + prompts)

Default: shallow-clone **`https://github.com/daveyous07/phillipinelawai`** (contains `backend/` at repo root). Or set `PHLAW_REPO_URL` / manual `BACKEND_ROOT`.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = os.environ.get(
    "PHLAW_REPO_URL",
    "https://github.com/daveyous07/phillipinelawai.git",
)
CLONE_DIR = Path("/content/phlaw-repo")


def find_backend(root: Path):
    """Monorepo: .../philippine-law-assistant/backend. Standalone: .../backend."""
    for rel in ("philippine-law-assistant/backend", "backend"):
        p = root / rel
        if (p / "app" / "prompts.py").is_file():
            return p
    return None


BACKEND_ROOT = find_backend(CLONE_DIR)
if BACKEND_ROOT is None:
    if not (CLONE_DIR / "backend" / "app" / "prompts.py").is_file():
        CLONE_DIR.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(CLONE_DIR)],
            check=False,
        )
    BACKEND_ROOT = find_backend(CLONE_DIR)

# --- Upload zip: uncomment and set ---
# BACKEND_ROOT = Path("/content/philippine-law-assistant/backend")

if BACKEND_ROOT is None:
    raise FileNotFoundError(
        "Could not find backend/app/prompts.py. Check git clone output, set PHLAW_REPO_URL, or set BACKEND_ROOT."
    )

BACKEND_ROOT = BACKEND_ROOT.resolve()
sys.path.insert(0, str(BACKEND_ROOT))

from app.prompts import PHILIPPINE_LAW_TUTOR_SYSTEM
from app.services.rag_simple import retrieve

print("Backend:", BACKEND_ROOT)

## 3) Load an instruct model on GPU

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = os.environ.get("PHLAW_MODEL_ID", "Qwen/Qwen2.5-1.5B-Instruct")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()
print("Loaded:", MODEL_ID)

## 4) Chat helper (RAG + generate)

In [ ]:
def build_context(docs):
    if not docs:
        return ""
    parts = ["Relevant provisions (cite these only):\n"]
    for r in docs:
        raw = str(r.get("content", ""))
        content = " ".join(raw.split())[:2000]
        parts.append(f"[{r['source']}]\n{content}\n")
    return "\n".join(parts)


def chat_colab(
    message: str,
    *,
    top_k: int = 5,
    max_new_tokens: int = 512,
    temperature: float = 0.6,
):
    docs = retrieve(message, top_k=top_k)
    ctx = build_context(docs)
    user = f"{ctx}\n\nUser question: {message}" if ctx else message
    messages = [
        {"role": "system", "content": PHILIPPINE_LAW_TUTOR_SYSTEM},
        {"role": "user", "content": user},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    gen_kw = {"max_new_tokens": max_new_tokens, "pad_token_id": tokenizer.eos_token_id}
    if temperature is not None and temperature > 0:
        gen_kw["do_sample"] = True
        gen_kw["temperature"] = temperature
    else:
        gen_kw["do_sample"] = False
    with torch.inference_mode():
        out = model.generate(**inputs, **gen_kw)
    gen = out[0, inputs["input_ids"].shape[1] :]
    return tokenizer.decode(gen, skip_special_tokens=True).strip()

In [ ]:
q = "What is due process under Article III of the 1987 Constitution?"
print(chat_colab(q))

## 5) Inspect RAG only (no LLM)

In [ ]:
for i, doc in enumerate(retrieve("writ of habeas corpus", top_k=3), 1):
    print(i, doc["source"], "—", doc["content"][:200].replace("\n", " "), "…")

## 6) One-click: public URL for the **Android app** (no signup)

After **sections 1–4** have run (model + `chat_colab` exists):

1. Run the **next two cells** (install FastAPI, then start server + **Cloudflare quick tunnel**).
2. Copy the **`https://….trycloudflare.com`** line the last cell prints.
3. In the app: **Settings** → **Backend API URL** → paste → **Save**. Leave Colab **running**.

No ngrok account needed. If the tunnel line does not appear, **Runtime → Restart runtime** and re-run from section 1.

In [ ]:
%pip install -q fastapi uvicorn pydantic

In [ ]:
import os
import re
import subprocess
import threading
import time
import urllib.request

import uvicorn
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import StreamingResponse
from pydantic import BaseModel

assert "chat_colab" in dir(), "Run sections 1–4 first so chat_colab is defined."


class ChatRequest(BaseModel):
    message: str
    history: list = []


app = FastAPI(title="PHLaw Colab")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)


@app.get("/health")
def health():
    return {"status": "ok"}


@app.post("/api/chat", response_model=None)
async def chat_api(req: ChatRequest):
    import asyncio

    if not req.message.strip():
        return {"response": ""}
    loop = asyncio.get_event_loop()
    text = await loop.run_in_executor(None, lambda: chat_colab(req.message.strip()))
    return {"response": text}


@app.post("/api/chat/stream")
async def chat_stream_api(req: ChatRequest):
    import asyncio

    if not req.message.strip():

        async def empty():
            yield "data: \n\n"

        return StreamingResponse(empty(), media_type="text/event-stream")

    msg = req.message.strip()

    async def gen():
        loop = asyncio.get_event_loop()
        text = await loop.run_in_executor(None, lambda: chat_colab(msg))
        yield f"data: {text}\n\n"

    return StreamingResponse(gen(), media_type="text/event-stream")


def _run_uvicorn():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")


threading.Thread(target=_run_uvicorn, daemon=True).start()
time.sleep(2)

# Cloudflare quick tunnel — no account (public https URL for your phone)
CF_BIN = "/tmp/cloudflared"
if not os.path.isfile(CF_BIN):
    url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
    urllib.request.urlretrieve(url, CF_BIN)
    os.chmod(CF_BIN, 0o755)

proc = subprocess.Popen(
    [CF_BIN, "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
public_url = None
buf = ""
deadline = time.time() + 90
while time.time() < deadline and public_url is None:
    line = proc.stdout.readline()
    if line:
        buf += line
        m = re.search(r"(https://[a-zA-Z0-9.-]+\.trycloudflare\.com)", buf)
        if m:
            public_url = m.group(1)
            break
    elif proc.poll() is not None:
        break
    else:
        time.sleep(0.05)

print()
print("=" * 62)
if public_url:
    print("  COPY THIS → App → Settings → Backend API URL → Save")
    print()
    print(" ", public_url)
    print()
    print("  Keep this Colab running while you use the app.")
else:
    print("  Tunnel URL not detected. Scroll up for cloudflared errors.")
    print("  Try: Runtime → Restart runtime, re-run from section 1.")
print("=" * 62)